# Surface ELO

In [10]:
import sys
from pathlib import Path

# Regarding Tennis_ML_Project/Notebooks, scr is a sibbling, not a child, 
# so we need to add the parent of Notebooks to the path so that we can import from src

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


In [11]:
import src
print("src imported correctly:", src)

src imported correctly: <module 'src' from 'c:\\Users\\X421IA\\Desktop\\Learning Projects\\Tennis_ML_Project\\src\\__init__.py'>


### Load and prepare Data

In [12]:
import pandas as pd

matches = pd.read_csv("../Data/processed/matches_with_players_clusters.csv")

#### Create ELO feature

In [13]:
from src.features import compute_surface_elo

matches = compute_surface_elo(matches)
matches.head()

{('Ryan Harrison', 'Hard'): 1470.0486636207368, ('Leonardo Mayer', 'Hard'): 1445.426016868708, ('Jared Donaldson', 'Hard'): 1495.0123918252386, ('Jordan Thompson', 'Hard'): 1693.1542395728288, ('Denis Istomin', 'Hard'): 1360.6990593364492, ('Damir Dzumhur', 'Hard'): 1385.35678203729, ('Alex De Minaur', 'Hard'): 1780.6828353124886, ('Steve Johnson', 'Hard'): 1467.4494762160061, ('Michael Mmoh', 'Hard'): 1585.3505681562986, ('Federico Delbonis', 'Hard'): 1313.5206095314327, ('Mischa Zverev', 'Hard'): 1371.8957456865405, ('John Patrick Smith', 'Hard'): 1420.1530091026648, ('Alexandr Dolgopolov', 'Hard'): 1524.8088281510052, ('Diego Schwartzman', 'Hard'): 1494.0915985662027, ('Horacio Zeballos', 'Hard'): 1470.0245424731256, ('Ernesto Escobedo', 'Hard'): 1416.366469966606, ('Matthew Ebden', 'Hard'): 1488.8079175848507, ('Frances Tiafoe', 'Hard'): 1682.6352895958926, ('Hyeon Chung', 'Hard'): 1557.5850955604824, ('Gilles Muller', 'Hard'): 1431.8504539215473, ('Kyle Edmund', 'Hard'): 1531.0307

,year,tourney_date,surface,tourney_level,winner_id,winner_name,winner_rank,winner_age,winner_ht,loser_id,...,w_bpSaved,w_bpFaced,l_ace,l_df,l_bpSaved,l_bpFaced,winner_cluster,loser_cluster,surface_elo_winner,surface_elo_loser
0,2018,20180101,Hard,A,105992,Ryan Harrison,47.0,25.6,185.0,104919,...,8.0,9.0,10.0,3.0,1.0,4.0,4,4,1500.0,1500.0
1,2018,20180101,Hard,A,111577,Jared Donaldson,54.0,21.2,188.0,111442,...,4.0,5.0,3.0,5.0,7.0,11.0,4,3,1500.0,1500.0
2,2018,20180101,Hard,A,104797,Denis Istomin,63.0,31.3,185.0,106000,...,9.0,11.0,8.0,6.0,10.0,16.0,1,3,1500.0,1500.0
3,2018,20180101,Hard,A,200282,Alex De Minaur,208.0,18.8,183.0,105449,...,2.0,3.0,6.0,2.0,4.0,6.0,0,4,1500.0,1500.0
4,2018,20180101,Hard,A,111581,Michael Mmoh,175.0,19.9,188.0,105643,...,3.0,3.0,4.0,0.0,0.0,2.0,4,1,1500.0,1500.0


In [17]:
# Let's check the ELO ratings for Alexander Zverev in the matches dataset
zverev_elo = matches[(matches["winner_name"] == "Alexander Zverev") | (matches["loser_name"] == "Alexander Zverev")]
zverev_elo[["winner_name", "loser_name", "surface", "surface_elo_winner", "surface_elo_loser"]].head(19)

,winner_name,loser_name,surface,surface_elo_winner,surface_elo_loser
186,Alexander Zverev,Thomas Fabbiano,Hard,1500.000000,1484.000000
226,Alexander Zverev,Peter Gojowczyk,Hard,1515.263693,1545.164684
246,Hyeon Chung,Alexander Zverev,Hard,1548.066068,1532.637295
294,Alexander Zverev,Alex De Minaur,Hard,1517.347350,1552.596433
296,Alexander Zverev,Nick Kyrgios,Hard,1534.965083,1597.217843
524,Alexander Zverev,David Ferrer,Hard,1553.801638,1499.943859
534,Andreas Seppi,Alexander Zverev,Hard,1527.387022,1567.341073
633,Alexander Zverev,Mackenzie Mcdonald,Hard,1549.509189,1483.994065
649,Alexander Zverev,Peter Gojowczyk,Hard,1562.527365,1576.094466
657,Alexander Zverev,Ryan Harrison,Hard,1579.151836,1560.037621


#### Build Dataset

In [19]:
def build_match_dataset_with_surface_elo(matches):
    rows = []

    for _, row in matches.iterrows():
        # winner vs loser
        rows.append({
            "year": row["year"],
            "rank_diff": row["winner_rank"] - row["loser_rank"],
            "age_diff": row["winner_age"] - row["loser_age"],
            "height_diff": row["winner_ht"] - row["loser_ht"],
            "cluster_diff": row["winner_cluster"] - row["loser_cluster"],
            "elo_diff": row["surface_elo_winner"] - row["surface_elo_loser"],
            "surface": row["surface"],
            "tourney_level": row["tourney_level"],
            "target": 1
        })

        # loser vs winner
        rows.append({
            "year": row["year"],  # we keep year
            "rank_diff": row["loser_rank"] - row["winner_rank"],
            "age_diff": row["loser_age"] - row["winner_age"],
            "height_diff": row["loser_ht"] - row["winner_ht"],
            "cluster_diff": row["loser_cluster"] - row["winner_cluster"],
            "elo_diff": row["surface_elo_loser"] - row["surface_elo_winner"],
            "surface": row["surface"],
            "tourney_level": row["tourney_level"],
            "target": 0
        })

    return pd.DataFrame(rows)

dataset = build_match_dataset_with_surface_elo(matches)
dataset.head()

,year,rank_diff,age_diff,height_diff,cluster_diff,elo_diff,surface,tourney_level,target
0,2018,-5.0,-5.0,-3.0,0,0.0,Hard,A,1
1,2018,5.0,5.0,3.0,0,0.0,Hard,A,0
2,2018,-40.0,-2.5,5.0,1,0.0,Hard,A,1
3,2018,40.0,2.5,-5.0,-1,0.0,Hard,A,0
4,2018,33.0,5.7,10.0,-2,0.0,Hard,A,1


#### Temporal Split

In [20]:
train_data = dataset[dataset["year"] <= 2022]
val_data = dataset[dataset["year"] == 2023]
test_data = dataset[dataset["year"] > 2023]

#### Imputation

In [21]:
from sklearn.impute import SimpleImputer

numeric_features = ["rank_diff", "age_diff", "height_diff", "cluster_diff", "elo_diff"]

num_imputer = SimpleImputer(strategy="median")
train_data[numeric_features] = num_imputer.fit_transform(train_data[numeric_features])
val_data[numeric_features] = num_imputer.transform(val_data[numeric_features])
test_data[numeric_features] = num_imputer.transform(test_data[numeric_features])

C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\2021079800.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[numeric_features] = num_imputer.fit_transform(train_data[numeric_features])
C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\2021079800.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_data[numeric_features] = num_imputer.transform(val_data[numeric_features])
C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\2021079800.py:8: SettingWithCopyWarning: 
A value is trying to be s

In [22]:
cat_imputer = SimpleImputer(strategy="most_frequent")

train_data[["surface"]] = cat_imputer.fit_transform(train_data[["surface"]])
val_data[["surface"]] = cat_imputer.transform(val_data[["surface"]])
test_data[["surface"]] = cat_imputer.transform(test_data[["surface"]])

C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\3314849990.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[["surface"]] = cat_imputer.fit_transform(train_data[["surface"]])
C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\3314849990.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_data[["surface"]] = cat_imputer.transform(val_data[["surface"]])
C:\Users\X421IA\AppData\Local\Temp\ipykernel_26308\3314849990.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [23]:
print(train_data.isna().sum())
print(val_data.isna().sum())
print(test_data.isna().sum())

year             0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
elo_diff         0
surface          0
tourney_level    0
target           0
dtype: int64
year             0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
elo_diff         0
surface          0
tourney_level    0
target           0
dtype: int64
year             0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
elo_diff         0
surface          0
tourney_level    0
target           0
dtype: int64


#### Encoding and Scaler

In [24]:
categorical_features = ["surface", "tourney_level"]
numeric_features = ["rank_diff", "age_diff",
                    "height_diff", "cluster_diff", "elo_diff"]


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"  # drop any other columns
)


In [26]:
X_train = train_data.drop(columns=["target"])
y_train = train_data["target"]

X_val = val_data.drop(columns=["target"])
y_val = val_data["target"]

X_test = test_data.drop(columns=["target"])
y_test = test_data["target"]

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


#### Return to Data Frame

In [27]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_val_df = pd.DataFrame(X_val_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

## Train Logistic Regression With ELO Surface

In [28]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_df, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [29]:
from sklearn.metrics import roc_auc_score, accuracy_score

val_probs = log_reg.predict_proba(X_val_df)[:, 1]
val_auc = roc_auc_score(y_val, val_probs)

print("Validation ROC-AUC with ELO:", val_auc)

Validation ROC-AUC with ELO: 0.7034749460420117


### Evaluate on Test

In [30]:
y_test_pred = log_reg.predict(X_test_df)
y_test_proba = log_reg.predict_proba(X_test_df)[:, 1]

test_acc = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"Test Accuracy (2024): {test_acc:.4f}")
print(f"Test ROC-AUC (2024): {test_auc:.4f}")

Test Accuracy (2024): 0.6466
Test ROC-AUC (2024): 0.7107


## Train Neural Network with ELO

In [31]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [32]:
X_train_t = torch.tensor(X_train_df.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32)

X_val_t = torch.tensor(X_val_df.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val.to_numpy(), dtype=torch.float32)

X_test_t = torch.tensor(X_test_df.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

print(X_train_t.shape, y_train_t.shape)


torch.Size([25630, 13]) torch.Size([25630])


#### Datasets and DataLoaders

In [33]:
from src.dataset import MatchDataset
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = MatchDataset(X_train_t, y_train_t)
val_dataset = MatchDataset(X_val_t, y_val_t)
test_dataset = MatchDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [34]:
from src.models import MatchOutcomeNN

input_dim = X_train_df.shape[1]
model = MatchOutcomeNN(input_dim).to(device)

model


MatchOutcomeNN(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)

#### Loss function and optimizer

In [35]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


### Training

In [36]:
from src.train import train_one_epoch, evaluate

EPOCHS = 20
best_val_auc = 0.0
best_state_dict = None


In [37]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(     # train using training data (2018-2022)
        model, 
        train_loader,
        optimizer, 
        criterion, 
        device)
    
    val_acc, val_auc, *_, = evaluate(      # evaluate on validation data (2023)
        model, 
        val_loader, 
        device)
    
    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    # Save best model (by AUC)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state_dict = model.state_dict()
    
    

Epoch 01 | Train Loss: 0.6459 | Val Acc: 0.6386 | Val AUC: 0.7032
Epoch 02 | Train Loss: 0.6299 | Val Acc: 0.6397 | Val AUC: 0.7054
Epoch 03 | Train Loss: 0.6275 | Val Acc: 0.6403 | Val AUC: 0.7056
Epoch 04 | Train Loss: 0.6275 | Val Acc: 0.6408 | Val AUC: 0.7053
Epoch 05 | Train Loss: 0.6247 | Val Acc: 0.6390 | Val AUC: 0.7056
Epoch 06 | Train Loss: 0.6248 | Val Acc: 0.6427 | Val AUC: 0.7070
Epoch 07 | Train Loss: 0.6233 | Val Acc: 0.6437 | Val AUC: 0.7070
Epoch 08 | Train Loss: 0.6219 | Val Acc: 0.6427 | Val AUC: 0.7065
Epoch 09 | Train Loss: 0.6240 | Val Acc: 0.6427 | Val AUC: 0.7062
Epoch 10 | Train Loss: 0.6227 | Val Acc: 0.6432 | Val AUC: 0.7064
Epoch 11 | Train Loss: 0.6226 | Val Acc: 0.6428 | Val AUC: 0.7070
Epoch 12 | Train Loss: 0.6225 | Val Acc: 0.6452 | Val AUC: 0.7077
Epoch 13 | Train Loss: 0.6222 | Val Acc: 0.6427 | Val AUC: 0.7076
Epoch 14 | Train Loss: 0.6221 | Val Acc: 0.6448 | Val AUC: 0.7075
Epoch 15 | Train Loss: 0.6210 | Val Acc: 0.6445 | Val AUC: 0.7072
Epoch 16 |

### Load Best Model

In [38]:
model.load_state_dict(best_state_dict)
print("Best validation AUC:", best_val_auc)


Best validation AUC: 0.7085081462991617


### Evaluate on Test set

In [39]:
test_acc, test_auc, *_ = evaluate(
    model,
    test_loader,
    device
)

print(f"Neural Network Test Accuracy (2024): {test_acc:.4f}")
print(f"Neural Network Test ROC-AUC (2024): {test_auc:.4f}")

Neural Network Test Accuracy (2024): 0.6481
Neural Network Test ROC-AUC (2024): 0.7116


## Comparison

#### Logistic Regression

| Feature Setup        | Test Accuracy | Test ROC-AUC |
| -------------------- | ------------- | ------------ |
| Global ELO           | 0.6466        | **0.7118**   |
| Surface-specific ELO | 0.6466        | 0.7107       |

---

#### Neural Network

| Feature Setup        | Test Accuracy | Test ROC-AUC |
| -------------------- | ------------- | ------------ |
| Global ELO           | 0.6484        | **0.7129**   |
| Surface-specific ELO | 0.6481        | 0.7116       |



Surface ELO is slightly worse. Global ELO is the better and more stable configuration. This tells us something important:

* Your dataset size (2018–2024) likely isn't large enough per surface
* Global ELO already captures most of the strength signal